# Task 2 — Validation of Temporal Knowledge Graphs (Google Colab / Jupyter)

Этот блокнот является **естественным продолжением Task 1**.

Эксперт загружает YAML-файл, созданный в первом блокноте, после чего блокнот автоматически:
- строит **эталонный (gold) граф**, который точно воспроизводит ручную reasoning-схему из YAML;
- при желании строит **автоматический temporal KG** по тем же статьям через пайплайн репозитория;
- сохраняет обе версии графа, таблицы триплетов, comparison summary и визуализации;
- показывает эксперту **текстовое представление триплетов** и **интерактивные визуализации** для ручной валидации.

Блокнот **не патчит файлы репозитория**. Он только ставит зависимости, импортирует уже готовый API репозитория и запускает workflow.


# Пошаговый туториал

## Шаг 1. Подготовьте репозиторий
Рекомендуемый вариант — распаковать архив `top-papers-graph-fixed-v3.zip` в `/content` или рядом с ноутбуком.

## Шаг 2. Запустите ячейку «Установка и импорт»
Она:
- найдёт локальный репозиторий автоматически;
- если локальный репозиторий не найден, клонирует официальный GitHub-репозиторий;
- установит зависимости для Task 2 notebook;
- импортирует функции Task 2.

## Шаг 3. Запустите ячейку «Вспомогательные функции»
Она подготовит UI, рендеринг таблиц и графов.

## Шаг 4. Запустите ячейку «Форма Task 2»
В ней нужно:
1. загрузить YAML из Task 1;
2. выбрать, строить ли автоматический граф;
3. нажать кнопку запуска.

## Что вы получите
- **Gold graph**: полностью повторяет reasoning trajectory эксперта;
- **Auto graph**: строится одной командой поверх статей из YAML;
- **Triplets tables**: CSV/JSON для ручной проверки;
- **Comparison summary**: базовое сравнение gold vs auto;
- **Reference scout**: дополнительные ссылки-кандидаты для примеров/контрпримеров.


In [ ]:
# @title
# Установка и импорт (запустите эту ячейку)
import json, os, sys, tempfile, subprocess
from pathlib import Path


def run(cmd, cwd=None):
    print('>', ' '.join(map(str, cmd)))
    subprocess.check_call(cmd, cwd=cwd)


def find_repo_root() -> Path | None:
    candidates = []
    env_dir = os.environ.get('TPG_REPO_DIR')
    if env_dir:
        candidates.append(Path(env_dir))
    cwd = Path.cwd()
    candidates.extend([cwd, cwd.parent, Path('/content')])
    for base in [Path('/content'), cwd, cwd.parent]:
        if not base.exists():
            continue
        for pattern in ('top-papers-graph-fixed-v3', 'top-papers-graph-fixed*', 'top-papers-graph-main', 'top-papers-graph'):
            candidates.extend(base.glob(pattern))
    seen = set()
    uniq = []
    for c in candidates:
        try:
            r = c.resolve()
        except Exception:
            r = c
        if str(r) in seen:
            continue
        seen.add(str(r))
        uniq.append(r)
    for c in uniq:
        if (c / 'pyproject.toml').exists() and (c / 'src' / 'scireason').exists():
            return c
    return None


REPO_DIR = find_repo_root()
REPO_URL = 'https://github.com/top-papers/top-papers-graph.git'
if REPO_DIR is None:
    REPO_DIR = Path('/content/top-papers-graph') if Path('/content').exists() else Path.cwd() / 'top-papers-graph'
    if not REPO_DIR.exists():
        run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)])

# UI / notebook dependencies
run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'ipywidgets', 'pyyaml', 'requests', 'pandas',
    'panel', 'hvplot', 'holoviews', 'bokeh', 'pyvis', 'jupyter_bokeh'
])

# Prefer the all-in-one notebook extra when available; otherwise fall back to official repo extras.
try:
    run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO_DIR}[task2_notebook]'])
except Exception:
    run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO_DIR}[temporal,mm]'])

import pandas as pd
import ipywidgets as W
from IPython.display import display, Markdown, HTML, clear_output

try:
    import panel as pn
    pn.extension()
except Exception:
    pn = None

try:
    import hvplot  # noqa: F401
    import hvplot.networkx  # noqa: F401
except Exception:
    pass

# Import fixed compatibility API when available.
try:
    from scireason.task2_validation import (
        build_task2_validation_bundle,
        load_task1_yaml,
        make_hvplot_payload,
    )
except Exception:
    # Fallback for official repo snapshots that only expose the new pipeline path.
    import shutil
    import yaml  # type: ignore
    from dataclasses import dataclass
    from scireason.pipeline.task2_validation import (
        build_reference_graph,
        prepare_task2_validation_bundle,
        resolve_papers_from_trajectory,
        suggest_link_candidates,
    )

    @dataclass
    class BundleResult:
        bundle_dir: Path
        manifest_path: Path

    def load_task1_yaml(path):
        p = Path(path)
        doc = yaml.safe_load(p.read_text(encoding='utf-8')) or {}
        if not isinstance(doc, dict):
            raise ValueError('Task 1 YAML must contain a top-level object.')
        return doc

    def _write_triplets_csv(json_path: Path, csv_path: Path):
        rows = json.loads(json_path.read_text(encoding='utf-8'))
        pd.DataFrame(rows).to_csv(csv_path, index=False)
        return csv_path

    def _networkx_from_payload(payload):
        import networkx as nx
        edges = payload.get('edges') or []
        directed = not any(not bool(e.get('directed', True)) for e in edges if isinstance(e, dict))
        G = nx.DiGraph() if directed else nx.Graph()
        for node in payload.get('nodes', []) or []:
            if not isinstance(node, dict):
                continue
            node_id = node.get('id') or node.get('term') or node.get('label')
            if node_id is None:
                continue
            attrs = dict(node)
            attrs.setdefault('label', attrs.get('label') or attrs.get('term') or str(node_id))
            G.add_node(str(node_id), **attrs)
        for edge in edges:
            if not isinstance(edge, dict):
                continue
            src = edge.get('source')
            tgt = edge.get('target') or edge.get('object')
            if src is None or tgt is None:
                continue
            if str(src) not in G:
                G.add_node(str(src), label=str(src))
            if str(tgt) not in G:
                G.add_node(str(tgt), label=str(tgt))
            G.add_edge(str(src), str(tgt), **dict(edge))
        return G

    def make_hvplot_payload(payload):
        G = _networkx_from_payload(payload)
        try:
            import hvplot.networkx as hvnx
            import networkx as nx
            pos = nx.spring_layout(G, seed=7)
            plot = hvnx.draw(G, pos, node_size=10, with_labels=False, arrowhead_length=0.015, width=950, height=650)
            return G, plot
        except Exception:
            return G, None

    def _write_graph_html(graph_json_path: Path, html_path: Path):
        from pyvis.network import Network
        payload = json.loads(graph_json_path.read_text(encoding='utf-8'))
        G = _networkx_from_payload(payload)
        net = Network(height='750px', width='100%', directed=True, notebook=False)
        net.barnes_hut()
        for node_id, attrs in G.nodes(data=True):
            label = attrs.get('label') or attrs.get('term') or str(node_id)
            title = '\n'.join(f"{k}: {v}" for k, v in attrs.items() if k != 'label')
            net.add_node(str(node_id), label=str(label)[:80], title=title)
        for src, tgt, attrs in G.edges(data=True):
            label = attrs.get('predicate') or attrs.get('label') or ''
            title = '\n'.join(f"{k}: {v}" for k, v in attrs.items())
            net.add_edge(str(src), str(tgt), label=str(label), title=title)
        net.write_html(str(html_path), notebook=False)
        return html_path

    def build_task2_validation_bundle(trajectory_path, *, out_dir, include_auto_pipeline=True, multimodal=True, enable_reference_scout=True, run_vlm=True, edge_mode='auto', max_papers=0, max_link_queries=4):
        trajectory_path = Path(trajectory_path)
        out_dir = Path(out_dir)
        doc = load_task1_yaml(trajectory_path)
        run_name = str(doc.get('submission_id') or trajectory_path.stem)
        bundle_dir = out_dir / run_name
        if include_auto_pipeline:
            bundle_dir = prepare_task2_validation_bundle(
                trajectory_path,
                out_dir=out_dir,
                include_multimodal=multimodal,
                run_vlm=run_vlm,
                edge_mode=edge_mode,
                suggest_links=enable_reference_scout,
                max_papers=max_papers,
                max_link_queries=max_link_queries,
            )
        else:
            bundle_dir.mkdir(parents=True, exist_ok=True)
            shutil.copy2(trajectory_path, bundle_dir / trajectory_path.name)
            reference_graph = build_reference_graph(doc)
            (bundle_dir / 'reference_graph.json').write_text(json.dumps(reference_graph, ensure_ascii=False, indent=2), encoding='utf-8')
            (bundle_dir / 'reference_triplets.json').write_text(json.dumps(reference_graph.get('triplets') or [], ensure_ascii=False, indent=2), encoding='utf-8')
            if enable_reference_scout:
                try:
                    resolved = resolve_papers_from_trajectory(doc)
                    suggestions = suggest_link_candidates(doc, known_papers=resolved, max_queries=max_link_queries)
                except Exception:
                    suggestions = []
                scout_dir = bundle_dir / 'scout'
                scout_dir.mkdir(parents=True, exist_ok=True)
                (scout_dir / 'suggested_links.json').write_text(json.dumps(suggestions, ensure_ascii=False, indent=2), encoding='utf-8')

        gold_graph_json = bundle_dir / 'reference_graph.json'
        gold_triplets_json = bundle_dir / 'reference_triplets.json'
        gold_triplets_csv = bundle_dir / 'reference_triplets.csv'
        gold_graph_html = bundle_dir / 'reference_graph.html'
        _write_triplets_csv(gold_triplets_json, gold_triplets_csv)
        _write_graph_html(gold_graph_json, gold_graph_html)
        auto_graph_json = bundle_dir / 'automatic_graph' / 'temporal_kg.json'
        auto_triplets_json = bundle_dir / 'automatic_triplets.json'
        auto_triplets_csv = bundle_dir / 'automatic_triplets.csv'
        auto_graph_html = bundle_dir / 'automatic_graph.html'
        if auto_triplets_json.exists():
            _write_triplets_csv(auto_triplets_json, auto_triplets_csv)
        if auto_graph_json.exists():
            _write_graph_html(auto_graph_json, auto_graph_html)
        manifest = {
            'topic': str(doc.get('topic') or ''),
            'bundle_dir': str(bundle_dir),
            'gold_graph': str(gold_graph_json),
            'gold_graph_html': str(gold_graph_html),
            'gold_triplets_csv': str(gold_triplets_csv),
            'manifest_version': 4,
        }
        if auto_graph_json.exists():
            manifest.update({
                'auto_run_dir': str(bundle_dir / 'automatic_graph'),
                'auto_graph_json': str(auto_graph_json),
                'auto_graph_html': str(auto_graph_html),
                'auto_triplets_csv': str(auto_triplets_csv),
            })
        comparison = bundle_dir / 'comparison_summary.json'
        if comparison.exists():
            manifest['comparison_summary'] = str(comparison)
        scout = bundle_dir / 'scout' / 'suggested_links.json'
        if scout.exists():
            manifest['reference_scout'] = str(scout)
        manifest_path = bundle_dir / 'task2_notebook_manifest.json'
        manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
        return BundleResult(bundle_dir=bundle_dir, manifest_path=manifest_path)

print('Готово. Репозиторий:', REPO_DIR)
print('Теперь запускайте следующую ячейку: «Вспомогательные функции».')


In [ ]:
# @title
# Вспомогательные функции (не нужно редактировать)
from pathlib import Path


def _save_uploaded_yaml(upload_value, out_dir: Path) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)
    if isinstance(upload_value, dict):
        name, meta = next(iter(upload_value.items()))
        content = meta['content']
        path = out_dir / name
        path.write_bytes(content)
        return path
    if isinstance(upload_value, tuple) and upload_value:
        meta = upload_value[0]
        name = meta.get('name') or 'task1.yaml'
        content = meta.get('content')
        path = out_dir / name
        path.write_bytes(bytes(content))
        return path
    raise ValueError('Не удалось прочитать загруженный файл. Загрузите YAML ещё раз.')


def _display_manifest(manifest_path: Path):
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    lines = [
        f"- **topic**: {manifest.get('topic')}",
        f"- **bundle_dir**: `{manifest.get('bundle_dir')}`",
        f"- **gold_graph**: `{manifest.get('gold_graph')}`",
        f"- **gold_triplets_csv**: `{manifest.get('gold_triplets_csv')}`",
    ]
    if manifest.get('auto_run_dir'):
        lines += [
            f"- **auto_run_dir**: `{manifest.get('auto_run_dir')}`",
            f"- **auto_graph_json**: `{manifest.get('auto_graph_json')}`",
            f"- **auto_triplets_csv**: `{manifest.get('auto_triplets_csv')}`",
        ]
    if manifest.get('comparison_summary'):
        lines.append(f"- **comparison_summary**: `{manifest.get('comparison_summary')}`")
    if manifest.get('reference_scout'):
        lines.append(f"- **reference_scout**: `{manifest.get('reference_scout')}`")
    display(Markdown('## Артефакты Task 2\n' + '\n'.join(lines)))
    return manifest


def _show_dataframe(csv_path: str, title: str, max_rows: int = 100):
    p = Path(csv_path)
    if not p.exists():
        display(Markdown(f'**{title}:** файл не найден'))
        return
    df = pd.read_csv(p)
    display(Markdown(f'## {title} ({len(df)} строк)'))
    display(df.head(max_rows))


def _show_graph(graph_json_path: str, html_path: str, title: str):
    display(Markdown(f'## {title}'))
    gj = Path(graph_json_path)
    hp = Path(html_path)
    if gj.exists():
        try:
            payload = json.loads(gj.read_text(encoding='utf-8'))
            try:
                _, plot = make_hvplot_payload(payload)
                if plot is not None:
                    display(plot)
            except Exception as e:
                display(Markdown(f'hvPlot не удалось построить: `{e}`'))
        except Exception as e:
            display(Markdown(f'Не удалось открыть JSON графа: `{e}`'))
    if hp.exists():
        try:
            display(HTML(hp.read_text(encoding='utf-8')))
        except Exception:
            display(Markdown(f'HTML-файл графа: `{hp}`'))


def _show_reference_scout(path: str):
    p = Path(path)
    if not p.exists():
        return
    scout = json.loads(p.read_text(encoding='utf-8'))
    rows = []
    if isinstance(scout, dict):
        for hit in scout.get('hits', []):
            for res in hit.get('results', []):
                rows.append({
                    'query': hit.get('query'),
                    'id': res.get('id'),
                    'title': res.get('title'),
                    'year': res.get('year'),
                    'url': res.get('url'),
                })
    elif isinstance(scout, list):
        for res in scout:
            if not isinstance(res, dict):
                continue
            rows.append({
                'query': ', '.join(res.get('trigger_queries') or []),
                'id': res.get('paper_id') or res.get('id'),
                'title': res.get('title'),
                'year': res.get('year'),
                'url': res.get('url'),
                'score': res.get('score'),
            })
    if rows:
        display(Markdown('## Reference scout'))
        display(pd.DataFrame(rows).head(100))


In [ ]:
# @title
# Форма Task 2 (запустите ячейку и заполните поля)

yaml_upload = W.FileUpload(accept='.yaml,.yml', multiple=False, description='Загрузить YAML')
out_dir = W.Text(value='runs/task2_validation', description='Out dir', layout=W.Layout(width='100%'))
run_auto = W.Checkbox(value=True, description='Строить автоматический temporal KG')
reference_scout = W.Checkbox(value=True, description='Сгенерировать reference scout')
multimodal = W.Checkbox(value=True, description='Использовать multimodal pipeline')
run_btn = W.Button(description='Построить Task 2 bundle', button_style='success')
status = W.HTML()
out = W.Output()


def _on_run(_):
    with out:
        clear_output()
        try:
            status.value = '<b>Запуск...</b>'
            workdir = Path(tempfile.mkdtemp(prefix='task2_notebook_'))
            trajectory_path = _save_uploaded_yaml(yaml_upload.value, workdir)
            task1 = load_task1_yaml(trajectory_path)
            display(Markdown(f"# Загружен YAML\n- **topic**: {task1.get('topic')}\n- **submission_id**: `{task1.get('submission_id')}`"))

            bundle = build_task2_validation_bundle(
                trajectory_path,
                out_dir=Path(out_dir.value),
                include_auto_pipeline=bool(run_auto.value),
                multimodal=bool(multimodal.value),
                enable_reference_scout=bool(reference_scout.value),
            )
            manifest = _display_manifest(bundle.manifest_path)

            _show_dataframe(manifest['gold_triplets_csv'], 'Gold triplets')
            _show_graph(manifest['gold_graph'], manifest['gold_graph_html'], 'Gold graph')

            if manifest.get('auto_triplets_csv'):
                _show_dataframe(manifest['auto_triplets_csv'], 'Auto triplets')
            if manifest.get('auto_graph_json') and manifest.get('auto_graph_html'):
                _show_graph(manifest['auto_graph_json'], manifest['auto_graph_html'], 'Auto graph')
            if manifest.get('comparison_summary'):
                cmp = json.loads(Path(manifest['comparison_summary']).read_text(encoding='utf-8'))
                display(Markdown('## Comparison summary'))
                display(pd.DataFrame([cmp]))
            if manifest.get('reference_scout'):
                _show_reference_scout(manifest['reference_scout'])

            status.value = '<span style="color:green"><b>Готово.</b></span>'
        except Exception as e:
            status.value = f'<span style="color:red"><b>Ошибка:</b> {e}</span>'
            raise


run_btn.on_click(_on_run)

display(W.VBox([
    W.HTML('<h2>Task 2 — Validation UI</h2>'),
    yaml_upload,
    out_dir,
    run_auto,
    multimodal,
    reference_scout,
    run_btn,
    status,
    out,
]))


# CLI-режим (тот же workflow без notebook)

Если удобнее запускать всё одной командой из терминала, используйте:

```bash
top-papers-graph task2-bundle \
  --trajectory path/to/task1.yaml \
  --out-dir runs/task2_validation \
  --multimodal \
  --suggest-links
```

Эквивалентный основной алиас из репозитория:

```bash
top-papers-graph prepare-task2-validation \
  --trajectory path/to/task1.yaml \
  --out-dir runs/task2_validation \
  --multimodal \
  --suggest-links
```


# FAQ

## Что считается эталонным графом?
Эталонный граф строится **только из YAML Task 1** и полностью повторяет шаги эксперта, их evidence и связи между шагами.

## Что считается автоматическим графом?
Автоматический граф строится из **списка статей в YAML** и запускает встроенный конвейер репозитория: статьи → ingestion → temporal KG → review assets → report.

## Что делать, если автоматический граф почти пустой?
Это ожидаемо, если PDF недоступны, в статьях мало структурируемого текста или temporal evidence неявен. В этом случае ориентируйтесь на:
- `reference_scout.json`
- triplets CSV
- `comparison_summary.json`
- review templates из auto pipeline

## Как интерпретировать temporal поля?
- `start_date` / `end_date` — когда связь наблюдается / извлечена;
- `valid_from` / `valid_to` — когда открытие считается валидным;
- `temporal_basis` / `time_source` — откуда взялось время.
